In [1]:
!pip install beautifulsoup4

  Using cached soupsieve-2.6-py3-none-any.whl.metadata (4.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 186.0/186.0 kB 3.0 MB/s eta 0:00:00a 0:00:01
Using cached soupsieve-2.6-py3-none-any.whl (36 kB)

[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import requests
import time


search_url = "https://www.reddit.com/r/canada/search.json?q=tariffs&restrict_sr=1"


headers = {
    "User-Agent": "Mozilla/5.0 (compatible; MyScraper/1.0)"
}

articles = []


total_start_time = time.time()


print("Fetching search results...")
search_start_time = time.time()
response = requests.get(search_url, headers=headers)
search_end_time = time.time()

if response.status_code == 200:
    search_data = response.json()

    posts = search_data["data"]["children"][:10]
    print(f"Found {len(posts)} posts from search results.\n")
else:
    print("Failed to retrieve search results. Status code:", response.status_code)
    exit()

print("Time to fetch search results: {:.2f} seconds".format(search_end_time - search_start_time))


articles_start_time = time.time()
for idx, post in enumerate(posts, start=1):
    post_data = post["data"]
    title = post_data.get("title", "No Title")
    permalink = post_data.get("permalink", "")

    post_url = "https://www.reddit.com" + permalink

    json_url = post_url + ".json"
    
    print(f"Processing post {idx}: {post_url}")
    
    post_response = requests.get(json_url, headers=headers)
    if post_response.status_code == 200:
        post_json = post_response.json()

        detailed_post = post_json[0]["data"]["children"][0]["data"]
        selftext = detailed_post.get("selftext", "")
        articles.append({
            "title": title,
            "url": post_url,
            "selftext": selftext
        })
    else:
        print(f"Failed to fetch JSON for post: {post_url} (Status Code: {post_response.status_code})")
    

    time.sleep(1)
articles_end_time = time.time()

print("\nTime to fetch article details: {:.2f} seconds".format(articles_end_time - articles_start_time))

# Benchmark: Total elapsed time.
total_end_time = time.time()
print("Total time for scraping process: {:.2f} seconds\n".format(total_end_time - total_start_time))


print("Scraped Articles:")
for article in articles:
    print(f"Title: {article['title']}")
    print(f"URL: {article['url']}")
    print("Selftext excerpt:")
    print(article['selftext'][:300] + "...\n")
    print("-" * 80)


Fetching search results...
Found 10 posts from search results.

Time to fetch search results: 0.58 seconds
Processing post 1: https://www.reddit.com/r/canada/comments/1j3059b/statement_by_the_prime_minister_on_unjustified_us/
Processing post 2: https://www.reddit.com/r/canada/comments/1ifmyka/canada_retaliating_for_trumps_tariffs_with_25_per/
Processing post 3: https://www.reddit.com/r/canada/comments/1j4agtz/jack_daniels_maker_says_canada_pulling_us_alcohol/
Processing post 4: https://www.reddit.com/r/canada/comments/1j3kgzs/its_done_its_gone_ontario_premier_doug_ford/
Processing post 5: https://www.reddit.com/r/canada/comments/1in1dyz/trump_threatens_canadian_cars_with_tariffs_up_to/
Processing post 6: https://www.reddit.com/r/canada/comments/1j5uiiy/trump_threatens_new_tariffs_on_canada_including/
Processing post 7: https://www.reddit.com/r/canada/comments/1ifeslf/us_tariffs_will_be_imposed_on_feb_4/
Processing post 8: https://www.reddit.com/r/canada/comments/1ifmdo2/hockey_fans_boo